# ElasticSearch Test

In [41]:
from elasticsearch import Elasticsearch
import pandas as pd
import datetime
from statsmodels.tsa.ar_model import AR

Create Elasticsearch queue-prediction index

In [42]:
# ignore 400 cause by IndexAlreadyExistsException when creating an index
es = Elasticsearch()
# es = Elasticsearch(
#       ['localhost'],
#       http_auth=(username, 'password'),
#       verify_certs=False,
#       scheme="https",
#       port=443,
# )
es.indices.create(index='queues-prediction', ignore=400) #can be ignored

{'error': {'root_cause': [{'type': 'resource_already_exists_exception',
    'reason': 'index [queues-prediction/D3O1OrPrRqGFs1v3O06EAw] already exists',
    'index_uuid': 'D3O1OrPrRqGFs1v3O06EAw',
    'index': 'queues-prediction'}],
  'type': 'resource_already_exists_exception',
  'reason': 'index [queues-prediction/D3O1OrPrRqGFs1v3O06EAw] already exists',
  'index_uuid': 'D3O1OrPrRqGFs1v3O06EAw',
  'index': 'queues-prediction'},
 'status': 400}

### Load from Queue Index

Match products queue

In [43]:
res = es.search(index="queues", body={"query": {
                                            "match": {
                                                "name" : "products"
                                                }}}, size=1000) #define size

Get _source Data

In [44]:
d = [elem['_source'] for elem in res['hits']['hits']]

In [45]:
for elem in d:
    del elem['items']
    del elem['querytime']

Build Dataframe

In [46]:
df = pd.DataFrame(d)

In [47]:
df.index = df["timestamp"]

In [48]:
df.index = pd.to_datetime(df.index, format='%Y-%m-%dT%H:%M:%S.%f%z').sort_values()

In [49]:
df.drop(columns=['timestamp', 'name', 'tier'], inplace=True)

Resample Data to 1min Interval 

In [50]:
df = df.resample('1T').mean()

In [51]:
df

,size
timestamp,
2019-12-28 05:45:00+00:00,32662.0
2019-12-28 05:46:00+00:00,32667.5
2019-12-28 05:47:00+00:00,32672.0
2019-12-28 05:48:00+00:00,32677.0
2019-12-28 05:49:00+00:00,NaN
...,...
2019-12-31 04:00:00+00:00,27097.0
2019-12-31 04:01:00+00:00,27103.0
2019-12-31 04:02:00+00:00,27109.0


In [52]:
df = df.fillna(0)

In [53]:
df.head()

,size
timestamp,
2019-12-28 05:45:00+00:00,32662.0
2019-12-28 05:46:00+00:00,32667.5
2019-12-28 05:47:00+00:00,32672.0
2019-12-28 05:48:00+00:00,32677.0
2019-12-28 05:49:00+00:00,0.0


## ML Model

#### Train/Test Split

In [13]:
pred_horizon = 20 #minutes

In [14]:
data = df['size']

#### AR

In [15]:
model_ar = AR(data)
model_ar_fit = model_ar.fit(maxlag= 10,ic='bic', trend='nc', method='cmle', maxiter=20)

In [16]:
pred_test = model_ar_fit.predict(start=len(data), end=len(data)+pred_horizon-1, dynamic=True)

Build Prediction Dataframe

In [17]:
time_stamps = pd.date_range(start=data.index[-1]+datetime.timedelta(minutes=1), periods=20, freq='T')

In [18]:
d = {'timestamp': time_stamps, 'size': pred_test}
pred_df = pd.DataFrame(data=d)

In [ ]:
pred_df['timestamp'] = pred_df.timestamp.map(lambda x: datetime.datetime.strftime(x, '%Y-%m-%dT%H:%M:%S.%f%z'))

In [19]:
pred_df

,timestamp,size
2019-12-30 03:20:00+00:00,2019-12-30 03:20:00+00:00,33132.737737
2019-12-30 03:21:00+00:00,2019-12-30 03:21:00+00:00,33007.947259
2019-12-30 03:22:00+00:00,2019-12-30 03:22:00+00:00,32883.626790
2019-12-30 03:23:00+00:00,2019-12-30 03:23:00+00:00,32759.774558
2019-12-30 03:24:00+00:00,2019-12-30 03:24:00+00:00,32636.388802
2019-12-30 03:25:00+00:00,2019-12-30 03:25:00+00:00,32513.467762
2019-12-30 03:26:00+00:00,2019-12-30 03:26:00+00:00,32391.009691
2019-12-30 03:27:00+00:00,2019-12-30 03:27:00+00:00,32269.012842
2019-12-30 03:28:00+00:00,2019-12-30 03:28:00+00:00,32147.475481
2019-12-30 03:29:00+00:00,2019-12-30 03:29:00+00:00,32026.395875


In [20]:
count = 0
for index, row in pred_df.iterrows():
    doc_data = {
        'timestamp': row['timestamp'],
        'tier' : 'pic',
        'name' : 'products',
    #     'querytime' : 0,
        'size' : row['size'],
    #     'items' : " ".join(items)
    }
    count += 1
    es.index('queues-prediction', body=doc_data)
    if count % 5 == 0:
        print(str(count) + " Elemente hochgeladen")

5 Elemente hochgeladen
10 Elemente hochgeladen
15 Elemente hochgeladen
20 Elemente hochgeladen
